<a href="https://colab.research.google.com/github/Ekrem-Guler/RAG-Based-Mathematical-Problem-Solver/blob/main/RAG_MATH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install chromadb

In [ ]:
import pandas as pd
import numpy as np
import chromadb
from datasets import load_dataset

In [ ]:
ds = load_dataset("EleutherAI/hendrycks_math", "counting_and_probability")
ds


In [ ]:
ds_geo = load_dataset("EleutherAI/hendrycks_math", "geometry")
ds_geo

In [ ]:
ds_alg = load_dataset("EleutherAI/hendrycks_math", "algebra")
ds_alg

In [ ]:
data_train = pd.read_csv("train.csv")
data_test = pd.read_csv("test.csv")
data1 =pd.concat([data_train,data_test])

In [ ]:
ds1= pd.DataFrame.from_dict(ds["train"])
ds2 = pd.DataFrame.from_dict(ds_geo["train"])
ds3 = pd.DataFrame.from_dict(ds_alg["train"])
ds4 = pd.DataFrame.from_dict(ds["test"])
ds5 = pd.DataFrame.from_dict(ds_geo["test"])
ds6 = pd.DataFrame.from_dict(ds_alg["test"])
ds7 = pd.concat([ds1,ds2,ds3,ds4,ds5,ds6,data1],ignore_index=True)
ds7

In [ ]:
sentences = [ds["train"][i]["solution"] for i in range(len(ds["train"]))]
sentences_prob = [ds["train"][i]["problem"] for i in range(len(ds["train"]))]

In [ ]:
sentences = [ds["train"][i]["solution"] for i in range(len(ds["train"]))]
sentences_prob = [ds["train"][i]["problem"] for i in range(len(ds["train"]))]

In [ ]:
sentences_test = [ds["test"][i]["solution"] for i in range(len(ds["test"]))]
sentences_test_prob = [ds["test"][i]["problem"] for i in range(len(ds["test"]))]

In [ ]:
sentences_geo = [ds_geo["train"][i]["solution"] for i in range(len(ds_geo["train"]))]
sentences_geo_prob = [ds_geo["train"][i]["problem"] for i in range(len(ds_geo["train"]))]

In [ ]:
sentences_geo_test = [ds_geo["test"][i]["solution"] for i in range(len(ds_geo["test"]))]
sentences_geo_test_prob = [ds_geo["test"][i]["problem"] for i in range(len(ds_geo["test"]))]

In [ ]:
sentences_alg = [ds_alg["train"][i]["solution"] for i in range(len(ds_alg["train"]))]
sentences_alg_prob = [ds_alg["train"][i]["problem"] for i in range(len(ds_alg["train"]))]

In [ ]:
sentences_alg_test = [ds_alg["test"][i]["solution"] for i in range(len(ds_alg["test"]))]
sentences_alg_test_prob = [ds_alg["test"][i]["problem"] for i in range(len(ds_alg["test"]))]

In [ ]:
sentences_extra = data1["problem"].to_list()
sentences_extra_prob = data1["solution"].to_list()
sentences_extra

In [ ]:
sentences = ds7["problem"].to_list()
sentences_solution = ds7["solution"].to_list()
print(len(sentences_solution))

In [ ]:
pip install mathematics_dataset

In [ ]:
import mathematics_dataset

In [ ]:
mathematics_dataset

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(sentences)
print(len(embeddings[0]))

In [ ]:
embeddings_geo = model.encode(sentences_geo)
print(len(embeddings_geo[0]))

In [ ]:
embeddings_alg = model.encode(sentences_alg)
print(len(embeddings_alg[0]))

In [ ]:
embeddings_prob = model.encode(sentences_prob)
print(len(embeddings[0]))

In [ ]:
embeddings_geo_prob = model.encode(sentences_geo_prob)
print(len(embeddings_geo[0]))

In [ ]:
embeddings_alg_prob = model.encode(sentences_alg_prob)
print(len(embeddings_alg[0]))

In [ ]:
embeddings_extra_prob = model.encode(sentences_extra_prob)
print(len(embeddings_extra_prob[0]))

In [ ]:

chroma_client = chromadb.Client()
collection_math = chroma_client.create_collection(name="math_collection2",configuration={"hnsw": {"space": "cosine"}})


In [ ]:
collection_math.add(
    documents=sentences[0:5461],
    ids = [str(i) for i in range(5461)],
    metadatas=[{"source": s} for s in sentences_solution[0:5461]],
    embeddings=embeddings[0:5461].tolist()
)


In [ ]:
collection_math.add(
    documents=sentences[5461:10922],
    ids = [str(i+5461) for i in range(5461)],
    metadatas=[{"source": s} for s in sentences_solution[5461:10922]],
    embeddings=embeddings[5461:10922].tolist()
)


In [ ]:
collection_math.add(
    documents=sentences[10922:16383],
    ids = [str(i+10922) for i in range(5461)],
    metadatas=[{"source": s} for s in sentences_solution[10922:16383]],
    embeddings=embeddings[10922:16383].tolist()
)


In [ ]:
collection_math.add(
    documents=sentences[16383:18023],
    ids = [str(i+16383) for i in range(1640)],
    metadatas=[{"source": s} for s in sentences_solution[16383:18023]],
    embeddings=embeddings[16383:18023].tolist()
)


In [ ]:
def prime_numbers(low_ran, high_ran):
  primes = []
  for i in range(low_ran, high_ran):
    if i > 1:
      for j in range(2, i):
        if (i % j) == 0:
          break
      else:
        primes.append(i)
  return primes
prime_numbers(1,100)

In [ ]:
results = collection_math.query(
    query_texts=[
        "I have 8 white 10 black ball,if I take one ball what is the probability of the white ball? "
    ],
    n_results=10,
    include=["embeddings", "documents", "distances","metadatas"]
)

retrieved_solution = results["metadatas"][0][0]["source"]
retrieved_question = results["documents"][0][0]
print(f"Retrieved Solution: {retrieved_solution}\n Retrieved Question: {retrieved_question}")
for i in range(len(results['ids'][0])):
    print(f"ID: {results['ids'][0][i]}")
    print(f"Distance: {results['distances'][0][i]}") # Lower is better!
    print(f"Content snippet: {results['documents'][0][i][:50]}...")

In [ ]:
from huggingface_hub import InferenceClient


prompt = f"""Use the following RAG system
Retrieved Question: {retrieved_question},
Retrieved Solution: {retrieved_solution},
Question:I have 8 white 10 black ball,if I take one ball what is the probability of the white ball?
Solution:
"""
client = InferenceClient()

completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
)
print(completion.choices[0].message.content)